In [1]:
!pip install transformers peft accelerate bitsandbytes gradio python-docx reportlab -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 92.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch, re, os, tempfile
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import gradio as gr
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.pagesizes import A4

MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct"
ADAPTER = "/content/drive/MyDrive/project/qwen32b-lora"

print("Загвар ачаалж байна...")
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok.pad_token = tok.eos_token

mdl = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
mdl = PeftModel.from_pretrained(mdl, ADAPTER, is_trainable=False)
mdl.eval()
print("Загвар бэлэн!")

SYS = (
    "Та Монголын уул уурхайн салбарын гэрээ боловсруулдаг хуулийн мэргэжилтэн. "
    "Гэрээний холбогдох хэсгийг стандарт бүтэц, хууль зүйн зөв нэр томьёо ашиглан, "
    "өгөгдсөн нөхцөлд нийцүүлэн үүсгэнэ. "
    "Гаралт заавал зөвхөн кирилл монгол хэл дээр байх ёстой. "
    "Латин үсэг, хятад үсэг ашиглахгүй."
)

SECS = [
    (1,"Нийтлэг үндэслэл"),
    (2,"Тусгай зөвшөөрөл эзэмшигчийн эрх, үүрэг"),
    (3,"Засаг даргын эрх, үүрэг"),
    (4,"Талуудын харилцаа"),
    (5,"Гэрээний хариуцлага, маргаан шийдвэрлэх"),
    (6,"Гэрээний хэрэгжилт"),
    (7,"Гэрээний хугацаа, хүчин төгөлдөр болох"),
    (8,"Бусад зүйл"),
]

def fix_num(text, n):
    out = []
    for l in text.split("\n"):
        l = l.strip()
        if not l: continue
        l = re.sub(r"^\d+\.\s+([А-ЯӨҮЁ])", f"{n}. \\1", l)
        l = re.sub(r"^\d+\.(\d+\.?(?:\d+\.?)*)\s", f"{n}.\\1 ", l)
        out.append(l)
    return "\n".join(out)

def clean(text):
    text = re.sub(r'[\u4e00-\u9fff]+', '', text)
    text = re.sub(r'[\u3040-\u309f\u30a0-\u30ff]+', '', text)
    return text

def gen_sec(n, name, conds):
    ct = "\n".join(f"- {k}: {v}" for k,v in conds.items() if v)
    msgs = [
        {"role":"system","content":SYS},
        {"role":"user","content":
         f'Дараах нөхцөлөөр уул уурхайн гэрээний "{n}. {name}" хэсгийг үүсгэ:\n\n'
         f'{ct}\n\nЗаалтыг {n}.1, {n}.2... гэж дугаарла. '
         f'Зөвхөн монгол кирилл хэлээр бич.'}
    ]
    txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(txt, return_tensors="pt", truncation=True, max_length=3000).to("cuda")
    with torch.inference_mode():
        out = mdl.generate(
            **inp,
            max_new_tokens=500,
            do_sample=False,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
    res = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    return fix_num(clean(res.strip()), n)

def make_word(text, pa, pb):
    doc = Document()
    t = doc.add_paragraph()
    r = t.add_run("УУЛ УУРХАЙН ГЭРЭЭ")
    r.bold = True
    r.font.size = Pt(16)
    t.alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph("")
    for l in text.split("\n"):
        if not l.strip(): continue
        p   = doc.add_paragraph()
        run = p.add_run(l)
        if l.strip() and l[0].isdigit() and ". " in l[:4] and not l[2].isdigit():
            run.bold = True
            run.font.size = Pt(13)
        else:
            run.font.size = Pt(11)
    doc.add_paragraph("")
    doc.add_paragraph("")
    doc.add_paragraph().add_run("ТАЛУУДЫН ГАРЫН ҮСЭГ").bold = True
    doc.add_paragraph("")
    doc.add_paragraph(f"А тал: {pa}")
    doc.add_paragraph("Гарын үсэг: ___________________________")
    doc.add_paragraph("Тамга:      ___________________________")
    doc.add_paragraph("Огноо: ........ он .... сар .... өдөр")
    doc.add_paragraph("")
    doc.add_paragraph(f"Б тал: {pb}")
    doc.add_paragraph("Гарын үсэг: ___________________________")
    doc.add_paragraph("Тамга:      ___________________________")
    doc.add_paragraph("Огноо: ........ он .... сар .... өдөр")
    path = os.path.join(tempfile.mkdtemp(), "гэрээ.docx")
    doc.save(path)
    return path

def make_pdf(text, pa, pb):
    path = os.path.join(tempfile.mkdtemp(), "гэрээ.pdf")
    doc = SimpleDocTemplate(path, pagesize=A4,
                            leftMargin=60, rightMargin=60,
                            topMargin=60, bottomMargin=60)
    fn  = "Helvetica"
    fnb = "Helvetica-Bold"
    st  = ParagraphStyle("t", fontName=fnb, fontSize=15, alignment=1, spaceAfter=20)
    sh  = ParagraphStyle("h", fontName=fnb, fontSize=12, spaceAfter=8, spaceBefore=12)
    sn  = ParagraphStyle("n", fontName=fn,  fontSize=10, leading=16, spaceAfter=5)
    sig = ParagraphStyle("s", fontName=fn,  fontSize=10, leading=16)
    sigb= ParagraphStyle("sb",fontName=fnb, fontSize=11)
    els = [Paragraph("УУЛ УУРХАЙН ГЭРЭЭ", st)]
    for l in text.split("\n"):
        if not l.strip():
            els.append(Spacer(1, 6))
            continue
        safe = l.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        if l.strip() and l[0].isdigit() and ". " in l[:4] and not l[2].isdigit():
            els.append(Paragraph(safe, sh))
        else:
            els.append(Paragraph(safe, sn))
    els += [
        Spacer(1, 30),
        Paragraph("ТАЛУУДЫН ГАРЫН ҮСЭГ", sigb),
        Spacer(1, 20),
        Table([
            [Paragraph(f"<b>А тал: {pa}</b>", sig),
             Paragraph(f"<b>Б тал: {pb}</b>", sig)],
            [Paragraph("Гарын үсэг: ___________________", sig),
             Paragraph("Гарын үсэг: ___________________", sig)],
            [Paragraph("Тамга: ___________________", sig),
             Paragraph("Тамга: ___________________", sig)],
            [Paragraph("Огноо: .... он .. сар .. өдөр", sig),
             Paragraph("Огноо: .... он .. сар .. өдөр", sig)],
        ], colWidths=[230, 230])
    ]
    doc.build(els)
    return path

def run_model(ctype, aimag, sum_, area, mineral, dur, pa, pb, extra,
              edited_text, progress=gr.Progress()):
    conds = {
        "Гэрээний төрөл": ctype, "Аймаг": aimag, "Сум": sum_,
        "Талбай (га)": area, "Ашигт малтмал": mineral,
        "Хугацаа (жил)": dur, "А тал": pa, "Б тал": pb,
    }
    full = ["="*52, "         УУЛ УУРХАЙН ГЭРЭЭ", "="*52, ""]
    for i, (n, name) in enumerate(SECS):
        progress((i+1)/len(SECS), desc=f"{n}/{len(SECS)}: {name}")
        full.append(gen_sec(n, name, conds))
        full.append("")
    text = "\n".join(full)
    if extra.strip():
        text += f"\n\n9. Нэмэлт заалт\n9.1. {extra}"
    return text, make_word(text, pa, pb), make_pdf(text, pa, pb)

def download_edited(edited_text, pa, pb):
    if not edited_text.strip():
        return None, None
    return make_word(edited_text, pa, pb), make_pdf(edited_text, pa, pb)

css = """
.gradio-container{max-width:1100px!important;margin:0 auto!important;
  font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif!important;}
.header-wrap{padding:1.75rem 0 1.25rem;border-bottom:1px solid #e5e7eb;margin-bottom:1.5rem;}
.header-title{font-size:20px;font-weight:600;color:#111827;margin:0 0 3px;}
.header-sub{font-size:13px;color:#6b7280;margin:0 0 10px;}
.section-title{font-size:11px!important;font-weight:600!important;
  letter-spacing:0.07em!important;text-transform:uppercase!important;
  color:#9ca3af!important;margin-bottom:10px!important;display:block;}
button.primary{background:#111827!important;color:#fff!important;
  border-radius:8px!important;font-size:14px!important;font-weight:500!important;
  border:none!important;padding:11px 0!important;width:100%!important;
  margin-top:6px!important;cursor:pointer!important;transition:opacity 0.15s!important;}
button.primary:hover{opacity:0.82!important;}
button.secondary{background:#f3f4f6!important;color:#111827!important;
  border-radius:8px!important;font-size:13px!important;font-weight:500!important;
  border:1px solid #e5e7eb!important;padding:8px 0!important;
  width:100%!important;margin-top:4px!important;cursor:pointer!important;}
.notice{margin-top:1rem;padding:11px 15px;background:#fffbeb;
  border-left:3px solid #f59e0b;border-radius:0 6px 6px 0;
  font-size:12px;color:#92400e;line-height:1.65;}
.dark .header-wrap{border-bottom-color:#2d2d2d;}
.dark .header-title{color:#f3f4f6;}
.dark .header-sub{color:#9ca3af;}
.dark button.primary{background:#f9fafb!important;color:#111!important;}
.dark button.secondary{background:#1f2937!important;color:#f9fafb!important;
  border-color:#374151!important;}
.dark .notice{background:#1c1a00;color:#fbbf24;border-left-color:#d97706;}
"""

with gr.Blocks(title="Уул Уурхайн Гэрээ Үүсгэгч") as demo:
    gr.HTML("""
    <div class="header-wrap">
      <button onclick="document.body.classList.toggle('dark');
        this.textContent=document.body.classList.contains('dark')?'☀️ Light':'🌙 Dark'"
        style="float:right;padding:5px 14px;border-radius:20px;
               border:1px solid #d1d5db;background:transparent;
               cursor:pointer;font-size:12px;font-weight:500">🌙 Dark</button>
      <p class="header-title">Уул уурхайн гэрээ үүсгэгч</p>
      <p class="header-sub">Хиймэл оюун ухаанд суурилсан гэрээ боловсруулах систем</p>
    </div>""")

    with gr.Row(equal_height=False):
        with gr.Column(scale=4, min_width=300):
            gr.HTML('<span class="section-title">Гэрээний нөхцөл</span>')
            ctype = gr.Dropdown(
                ["Бичил уурхай","Ашиглалтын тухай гэрээ",
                 "Хайгуулын гэрээ","Хамтын ажиллагааны гэрээ"],
                value="Бичил уурхай", label="Гэрээний төрөл")
            with gr.Row():
                aimag = gr.Textbox(label="Аймаг", value="Сүхбаатар")
                sum_  = gr.Textbox(label="Сум",   value="Тариалан")
            with gr.Row():
                area    = gr.Textbox(label="Талбай (га)",    value="0.5")
                mineral = gr.Textbox(label="Ашигт малтмал", value="Жонш")
            dur = gr.Textbox(label="Хугацаа (жил)", value="1")
            with gr.Row():
                pa = gr.Textbox(label="А тал", value="Засаг дарга")
                pb = gr.Textbox(label="Б тал", value="БАЙГУУЛЛАГА нөхөрлөл")
            extra = gr.Textbox(
                label="Нэмэлт заалт (заавал биш)",
                placeholder="Жишээ: Орон нутгийн 5 иргэнийг ажиллуулна",
                lines=2)
            btn = gr.Button("Гэрээ үүсгэх", variant="primary")

        with gr.Column(scale=6, min_width=400):
            gr.HTML('<span class="section-title">Үүсгэсэн гэрээ — засварлах боломжтой</span>')
            out = gr.Textbox(
                label="", lines=24, max_lines=40, interactive=True,
                placeholder="Гэрээний нөхцөлийг бөглөөд үүсгэх товчийг дарна уу...")
            gr.HTML('<span class="section-title" '
                    'style="margin-top:10px;display:block;">Татаж авах</span>')
            with gr.Row():
                wf = gr.File(label="Word (.docx)")
                pf = gr.File(label="PDF (.pdf)")
            dl_btn = gr.Button("Засварласан гэрээг татаж авах", variant="secondary")
            with gr.Row():
                wf2 = gr.File(label="Засварласан Word")
                pf2 = gr.File(label="Засварласан PDF")
            gr.HTML(
                '<div class="notice"><strong>Анхааруулга:</strong> '
                'Энэхүү гэрээ нь AI-аар автомат үүсгэсэн төсөл бөгөөд '
                'хуулийн мэргэжилтнээр заавал хянуулах ёстой.</div>')

    btn.click(
        fn=run_model,
        inputs=[ctype,aimag,sum_,area,mineral,dur,pa,pb,extra,out],
        outputs=[out,wf,pf])
    dl_btn.click(
        fn=download_edited,
        inputs=[out,pa,pb],
        outputs=[wf2,pf2])

demo.launch(share=True)

Загвар ачаалж байна...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Загвар бэлэн!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a0b8899c56336aad7a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
